# Pre-entrega 5 - Pipeline ETL de datos

## Objetivo

El objetivo de esta entrega es diseñar e implementar un pipeline de datos que automatice las etapas de extracción, transformación y carga (ETL).

Para el desarrollo se utilizará Python y Pandas, separando la lógica del procesamiento en funciones reutilizables. El pipeline permitirá leer los datos de origen, realizar tareas de limpieza y transformación y generar finalmente un archivo procesado en formato Parquet.

La solución se organizará de manera modular, separando el archivo encargado de ejecutar el pipeline (`main.py`) de la lógica de procesamiento (`src/pipeline.py`).

Las etapas desarrolladas serán:

1. **Extracción:** lectura del dataset y revisión de sus tipos de datos.
2. **Transformación:** normalización de nombres de columnas, eliminación de registros duplicados y tratamiento de valores nulos.
3. **Carga:** almacenamiento del dataset procesado en formato Parquet.
4. **Validación:** comprobación del resultado generado por el pipeline.

## 1. Preparación del entorno de trabajo

En primer lugar, se prepara el entorno en Google Colab y se conecta Google Drive para disponer de un espacio persistente donde almacenar los archivos del proyecto.

Se creará una estructura simple que permita separar los datos de entrada, la lógica del pipeline y los archivos generados como resultado.

In [29]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Creación de la estructura del proyecto

Para evitar desarrollar un script monolítico, el proyecto se organizará separando el archivo de ejecución de la lógica del pipeline.

La estructura utilizada será:

preentrega-5-pipeline-etl/
- main.py
- requirements.txt
- README.md
- .gitignore
- src/
  - __init__.py
  - pipeline.py
- data/
- output/

La carpeta `data` contendrá el archivo original utilizado como entrada, mientras que `output` almacenará el archivo Parquet generado por el pipeline.

In [30]:
from pathlib import Path

PROJECT_PATH = Path("/content/drive/MyDrive/Data Science/Data Science II/preentrega-5-pipeline-etl")

PROJECT_PATH.mkdir(parents=True, exist_ok=True)

folders = [
    PROJECT_PATH / "src",
    PROJECT_PATH / "data",
    PROJECT_PATH / "output"
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Estructura del proyecto creada correctamente.")
print(f"Ubicación: {PROJECT_PATH}")

Estructura del proyecto creada correctamente.
Ubicación: /content/drive/MyDrive/Data Science/Data Science II/preentrega-5-pipeline-etl


## 3. Carga del dataset original

Antes de desarrollar el pipeline es necesario analizar la estructura del dataset de origen.

En esta etapa se carga el archivo CSV que posteriormente será utilizado por la función de extracción. El objetivo es conocer sus dimensiones, nombres de columnas y características generales antes de definir las transformaciones.

In [31]:
from google.colab import files

uploaded = files.upload()

Saving Absenteeism-data.csv to Absenteeism-data.csv


In [32]:
import shutil

uploaded_file = list(uploaded.keys())[0]
destination = PROJECT_PATH / "data" / "dataset.csv"

shutil.copy(uploaded_file, destination)

print(f"Dataset almacenado en: {destination}")

Dataset almacenado en: /content/drive/MyDrive/Data Science/Data Science II/preentrega-5-pipeline-etl/data/dataset.csv


## 4. Exploración inicial del dataset

Antes de automatizar el proceso ETL se realiza una revisión inicial de los datos.

Esta exploración permitirá:

- conocer la cantidad de registros y variables;
- revisar los nombres de las columnas;
- identificar los tipos de datos;
- detectar valores nulos;
- identificar registros duplicados.

Esta información será utilizada posteriormente para definir las reglas de transformación del pipeline.

In [33]:
import pandas as pd

dataset_path = PROJECT_PATH / "data" / "dataset.csv"

df = pd.read_csv(
    dataset_path,
    sep=None,
    engine="python"
)

print(f"Cantidad de filas: {df.shape[0]}")
print(f"Cantidad de columnas: {df.shape[1]}")

display(df.head())

Cantidad de filas: 700
Cantidad de columnas: 12


,ID,Reason for Absence,Date,Transportation Expense,Distance to Work,Age,Daily Work Load Average,Body Mass Index,Education,Children,Pets,Absenteeism Time in Hours
0,11,26,7/7/2015,289,36,33,239.554,30,1,2,1,4
1,36,0,14/7/2015,118,13,50,239.554,31,1,1,0,0
2,3,23,15/7/2015,179,51,38,239.554,31,1,0,0,2
3,7,7,16/7/2015,279,5,39,239.554,24,1,2,0,4
4,11,23,23/7/2015,289,36,33,239.554,30,1,2,1,2


### 4.1 Análisis de tipos de datos

Una parte importante de la etapa de extracción consiste en utilizar tipos de datos adecuados.

Pandas puede asignar automáticamente tipos de datos de mayor tamaño del necesario. Por este motivo se revisarán los tipos detectados para evaluar posteriormente si es posible optimizar el consumo de memoria.

Esta optimización cobra mayor importancia a medida que aumenta el volumen de información procesada.

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ID                         700 non-null    int64  
 1   Reason for Absence         700 non-null    int64  
 2   Date                       700 non-null    object 
 3   Transportation Expense     700 non-null    int64  
 4   Distance to Work           700 non-null    int64  
 5   Age                        700 non-null    int64  
 6   Daily Work Load Average    700 non-null    float64
 7   Body Mass Index            700 non-null    int64  
 8   Education                  700 non-null    int64  
 9   Children                   700 non-null    int64  
 10  Pets                       700 non-null    int64  
 11  Absenteeism Time in Hours  700 non-null    int64  
dtypes: float64(1), int64(10), object(1)
memory usage: 65.8+ KB


In [35]:
display(
    df.dtypes
    .rename("tipo_dato")
    .to_frame()
)

,tipo_dato
ID,int64
Reason for Absence,int64
Date,object
Transportation Expense,int64
Distance to Work,int64
Age,int64
Daily Work Load Average,float64
Body Mass Index,int64
Education,int64
Children,int64


### 4.2 Análisis de valores nulos

Los valores faltantes pueden producir problemas durante el análisis y el entrenamiento posterior de modelos de Machine Learning.

Por este motivo se analizará la cantidad de valores nulos por variable antes de establecer una estrategia de tratamiento.

La decisión de eliminar o imputar valores no debe realizarse de forma automática, sino considerando el significado de las variables y la cantidad de información faltante.

In [36]:
nulos = pd.DataFrame({
    "cantidad_nulos": df.isnull().sum(),
    "porcentaje_nulos": (df.isnull().mean() * 100).round(2)
})

display(nulos)

,cantidad_nulos,porcentaje_nulos
ID,0,0.0
Reason for Absence,0,0.0
Date,0,0.0
Transportation Expense,0,0.0
Distance to Work,0,0.0
Age,0,0.0
Daily Work Load Average,0,0.0
Body Mass Index,0,0.0
Education,0,0.0
Children,0,0.0


In [37]:
df.isnull().sum()

,0
ID,0
Reason for Absence,0
Date,0
Transportation Expense,0
Distance to Work,0
Age,0
Daily Work Load Average,0
Body Mass Index,0
Education,0
Children,0


### 4.3 Detección de registros duplicados

También se verificará la existencia de registros completamente duplicados.

Los duplicados exactos pueden generar sesgos en el análisis al representar más de una vez la misma observación. Por este motivo, la etapa de transformación del pipeline incorporará una regla para eliminarlos.

In [38]:
cantidad_duplicados = df.duplicated().sum()

print(f"Cantidad de registros duplicados: {cantidad_duplicados}")

Cantidad de registros duplicados: 8


### 4.4 Revisión de nombres de columnas

Finalmente se revisan los nombres originales de las variables.

Como parte de la transformación se normalizarán los nombres para obtener una estructura consistente, utilizando minúsculas y guiones bajos y eliminando espacios o caracteres innecesarios.

Esto facilita posteriormente la manipulación de las variables desde Python.


In [39]:
for columna in df.columns:
    print(columna)

ID
Reason for Absence
Date
Transportation Expense
Distance to Work
Age
Daily Work Load Average
Body Mass Index
Education
Children
Pets
Absenteeism Time in Hours


## 5. Conclusiones de la exploración inicial

El dataset analizado contiene **700 registros y 12 variables** relacionadas con características personales, laborales y de ausentismo.

A partir de la exploración inicial se identificaron los siguientes aspectos:

- El dataset no presenta valores nulos en ninguna de sus variables.
- Se detectaron **8 registros duplicados**, que deberán ser eliminados durante la etapa de transformación.
- La mayoría de las variables numéricas fueron interpretadas inicialmente como `int64`, mientras que `Daily Work Load Average` fue identificada como `float64`.
- La variable `Date` fue interpretada como `object`, por lo que será convertida a un formato de fecha apropiado.
- Los nombres de las columnas contienen espacios y combinaciones de mayúsculas y minúsculas. Para facilitar su manipulación serán normalizados utilizando minúsculas y guiones bajos.

Aunque el dataset actual no contiene valores faltantes, el pipeline incorporará una estrategia para gestionarlos en caso de que aparezcan en nuevas ejecuciones. De esta forma, el proceso será más robusto frente a futuros datos.

A partir de estos resultados se desarrollará un pipeline ETL que automatice las tareas de extracción, transformación y carga de los datos.

## 6. Desarrollo del pipeline ETL

A partir de los resultados obtenidos en la exploración inicial se desarrollará un pipeline ETL (Extract, Transform, Load) para automatizar el procesamiento del dataset.

Para evitar un script monolítico, la solución se dividirá en dos archivos principales:

- `main.py`: será el punto de entrada encargado de iniciar la ejecución.
- `src/pipeline.py`: contendrá las funciones responsables de la extracción, transformación y carga de los datos.

El proceso estará dividido en tres etapas:

### Extracción
Se realizará la lectura del archivo CSV y se optimizarán los tipos de datos utilizados.

### Transformación
Se normalizarán los nombres de las columnas, se convertirá la variable de fecha, se eliminarán registros duplicados y se gestionarán posibles valores nulos.

### Carga
El resultado procesado se almacenará en formato Parquet para obtener un archivo eficiente y adecuado para procesos posteriores de análisis y Machine Learning.

### 6.1 Extracción y optimización de tipos de datos

La exploración mostró que Pandas interpreta automáticamente la mayoría de las variables enteras como `int64`.

Debido a que los valores de varias variables se encuentran dentro de rangos pequeños, es posible utilizar tipos numéricos de menor tamaño. Esto permite reducir el consumo de memoria sin modificar la información almacenada.

En la función de extracción se utilizarán tipos de datos adecuados para las variables numéricas y la variable `Date` será convertida posteriormente a formato fecha.

Esta optimización busca mejorar la eficiencia del pipeline, especialmente pensando en escenarios donde el volumen de datos pueda aumentar.

In [40]:
pipeline_code = '''
from pathlib import Path
import re
import pandas as pd


def extraer_datos(ruta_entrada):
    """
    Lee el archivo CSV de entrada y devuelve un DataFrame.
    Se definen tipos de datos más eficientes para reducir
    el consumo de memoria.
    """

    tipos = {
        "ID": "int16",
        "Reason for Absence": "int8",
        "Transportation Expense": "int16",
        "Distance to Work": "int8",
        "Age": "int8",
        "Daily Work Load Average": "float32",
        "Body Mass Index": "int8",
        "Education": "int8",
        "Children": "int8",
        "Pets": "int8",
        "Absenteeism Time in Hours": "int16"
    }

    df = pd.read_csv(
        ruta_entrada,
        dtype=tipos
    )

    print(f"Extracción completada: {df.shape[0]} filas y {df.shape[1]} columnas.")

    return df


def normalizar_nombre(nombre):
    """
    Normaliza los nombres de las columnas utilizando
    minúsculas y guiones bajos.
    """

    nombre = nombre.strip().lower()
    nombre = re.sub(r"[^a-z0-9]+", "_", nombre)

    return nombre.strip("_")


def transformar_datos(df):
    """
    Aplica las tareas de limpieza y transformación.
    """

    df = df.copy()

    # 1. Normalización de nombres de columnas
    df.columns = [
        normalizar_nombre(columna)
        for columna in df.columns
    ]

    # 2. Conversión de la variable Date
    df["date"] = pd.to_datetime(
        df["date"],
        format="%d/%m/%Y",
        errors="coerce"
    )

    # 3. Eliminación de registros duplicados
    duplicados = df.duplicated().sum()
    df = df.drop_duplicates().copy()

    # 4. Tratamiento de valores nulos
    columnas_numericas = df.select_dtypes(
        include="number"
    ).columns

    for columna in columnas_numericas:
        if df[columna].isna().any():
            mediana = df[columna].median()
            df[columna] = df[columna].fillna(mediana)

    # Si una fecha no pudo convertirse, se elimina el registro
    nulos_fecha = df["date"].isna().sum()

    if nulos_fecha > 0:
        df = df.dropna(subset=["date"]).copy()

    print(f"Duplicados eliminados: {duplicados}")
    print(f"Registros con fecha inválida eliminados: {nulos_fecha}")
    print(f"Valores nulos restantes: {df.isna().sum().sum()}")

    return df


def cargar_datos(df, ruta_salida):
    """
    Exporta el DataFrame procesado a formato Parquet.
    """

    ruta_salida = Path(ruta_salida)

    ruta_salida.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    df.to_parquet(
        ruta_salida,
        engine="pyarrow",
        index=False
    )

    print(f"Archivo Parquet generado en: {ruta_salida}")


def ejecutar_pipeline(ruta_entrada, ruta_salida):
    """
    Ejecuta secuencialmente las etapas ETL.
    """

    print("Iniciando pipeline ETL...")

    df = extraer_datos(ruta_entrada)
    df = transformar_datos(df)
    cargar_datos(df, ruta_salida)

    print("Pipeline ETL finalizado correctamente.")
'''

pipeline_path = PROJECT_PATH / "src" / "pipeline.py"

with open(pipeline_path, "w", encoding="utf-8") as archivo:
    archivo.write(pipeline_code)

print(f"Archivo creado correctamente: {pipeline_path}")

Archivo creado correctamente: /content/drive/MyDrive/Data Science/Data Science II/preentrega-5-pipeline-etl/src/pipeline.py


### 6.2 Transformación y limpieza

La función `transformar_datos()` concentra las reglas de limpieza identificadas durante la exploración.

Se aplican las siguientes transformaciones:

1. **Normalización de columnas:** los nombres se convierten a minúsculas y los espacios son reemplazados por guiones bajos.

2. **Conversión de fecha:** la variable `Date`, originalmente interpretada como texto, se convierte al tipo de dato fecha.

3. **Eliminación de duplicados:** se eliminan registros completamente repetidos. En la exploración inicial se identificaron 8 registros duplicados.

4. **Tratamiento de valores nulos:** aunque el dataset actual no presenta datos faltantes, se incorpora una regla para manejar posibles nulos en futuras ejecuciones. Para las variables numéricas se utiliza la mediana porque es una medida robusta frente a valores extremos.

5. **Validación de fechas:** si un valor no puede convertirse correctamente a fecha, se considera inválido y el registro es eliminado.

Estas reglas permiten que la transformación pueda ejecutarse automáticamente cada vez que se incorporen nuevos datos.

#### Justificación del tratamiento de valores nulos

Se decidió utilizar la mediana para completar posibles valores faltantes en las variables numéricas.

A diferencia de la media, la mediana es menos sensible a valores extremos, por lo que permite completar datos faltantes reduciendo el efecto que podrían producir observaciones atípicas.

En el dataset actual no se detectaron valores nulos. Sin embargo, esta lógica se mantiene dentro del pipeline para que el proceso pueda gestionar automáticamente datos faltantes si aparecen en futuras ejecuciones.

## 7. Archivo de ejecución del pipeline

Para separar la ejecución de la lógica de procesamiento se utilizará un archivo `main.py`.

Este archivo no contendrá las reglas de transformación. Su responsabilidad será importar la función principal desde `src/pipeline.py` e iniciar el proceso ETL indicando las rutas de entrada y salida.

Esta separación favorece la modularización y facilita el mantenimiento y reutilización del código.

In [41]:
main_code = '''
from src.pipeline import ejecutar_pipeline


if __name__ == "__main__":
    ejecutar_pipeline(
        ruta_entrada="data/dataset.csv",
        ruta_salida="output/datos_procesados.parquet"
    )
'''

main_path = PROJECT_PATH / "main.py"

with open(main_path, "w", encoding="utf-8") as archivo:
    archivo.write(main_code)

print(f"Archivo creado correctamente: {main_path}")

Archivo creado correctamente: /content/drive/MyDrive/Data Science/Data Science II/preentrega-5-pipeline-etl/main.py


## 8. Dependencias del proyecto

Para garantizar la reproducibilidad del proyecto se creará un archivo `requirements.txt` con las librerías necesarias.

Se utilizarán:

- `pandas`: para la extracción, transformación y manipulación de los datos.
- `pyarrow`: como motor para almacenar y leer archivos en formato Parquet.

De esta manera, otra persona podrá instalar las dependencias necesarias antes de ejecutar el pipeline.

In [42]:
requirements = """pandas
pyarrow
"""

with open(PROJECT_PATH / "requirements.txt", "w") as archivo:
    archivo.write(requirements)

print("requirements.txt creado correctamente.")

requirements.txt creado correctamente.


## 9. Ejecución del pipeline

Una vez creados los módulos se ejecutará `main.py`.

Esto permitirá comprobar el funcionamiento integrado de las tres etapas:

**CSV → Extracción → Transformación → Carga → Parquet**

La ejecución deberá informar la cantidad de registros extraídos, los duplicados eliminados, los posibles valores nulos encontrados y la ubicación del archivo Parquet resultante.

### 9.1 Corrección de la lectura del archivo CSV

Durante la primera ejecución del pipeline se detectó que el archivo CSV fue interpretado como una única columna, a pesar de que el dataset original contiene 12 variables.

Esto se debe a que el separador utilizado por el archivo no coincide con el separador predeterminado de `pandas.read_csv()`.

Para solucionar este inconveniente se modificará la función de extracción para detectar automáticamente el separador del archivo, de la misma manera utilizada durante la exploración inicial.

Esta corrección garantiza que las 12 variables sean identificadas correctamente antes de iniciar las transformaciones.

In [43]:
pipeline_code = '''
from pathlib import Path
import re
import pandas as pd


def extraer_datos(ruta_entrada):
    """
    Lee el archivo CSV de entrada utilizando detección
    automática del separador.
    """

    df = pd.read_csv(
        ruta_entrada,
        sep=None,
        engine="python"
    )

    print(
        f"Extracción completada: "
        f"{df.shape[0]} filas y {df.shape[1]} columnas."
    )

    return df


def normalizar_nombre(nombre):
    """
    Normaliza los nombres de columnas utilizando
    minúsculas y guiones bajos.
    """

    nombre = nombre.strip().lower()
    nombre = re.sub(r"[^a-z0-9]+", "_", nombre)

    return nombre.strip("_")


def transformar_datos(df):
    """
    Realiza la limpieza y transformación de los datos.
    """

    df = df.copy()

    # Normalización de nombres de columnas
    df.columns = [
        normalizar_nombre(columna)
        for columna in df.columns
    ]

    # Conversión de la fecha
    df["date"] = pd.to_datetime(
        df["date"],
        format="%d/%m/%Y",
        errors="coerce"
    )

    # Eliminación de duplicados
    duplicados = df.duplicated().sum()
    df = df.drop_duplicates().copy()

    # Tratamiento de valores nulos numéricos
    columnas_numericas = df.select_dtypes(
        include="number"
    ).columns

    for columna in columnas_numericas:
        if df[columna].isna().any():
            mediana = df[columna].median()
            df[columna] = df[columna].fillna(mediana)

    # Eliminación de fechas inválidas
    fechas_invalidas = df["date"].isna().sum()

    if fechas_invalidas > 0:
        df = df.dropna(subset=["date"]).copy()

    # Optimización de tipos numéricos
    columnas_enteras = df.select_dtypes(include=["int64"]).columns
    columnas_decimales = df.select_dtypes(include=["float64"]).columns

    for columna in columnas_enteras:
        df[columna] = pd.to_numeric(
            df[columna],
            downcast="integer"
        )

    for columna in columnas_decimales:
        df[columna] = pd.to_numeric(
            df[columna],
            downcast="float"
        )

    print(f"Duplicados eliminados: {duplicados}")
    print(f"Fechas inválidas eliminadas: {fechas_invalidas}")
    print(f"Valores nulos restantes: {df.isna().sum().sum()}")

    return df


def cargar_datos(df, ruta_salida):
    """
    Guarda los datos procesados en formato Parquet.
    """

    ruta_salida = Path(ruta_salida)

    ruta_salida.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    df.to_parquet(
        ruta_salida,
        engine="pyarrow",
        index=False
    )

    print(f"Archivo Parquet generado en: {ruta_salida}")


def ejecutar_pipeline(ruta_entrada, ruta_salida):
    """
    Ejecuta las etapas de extracción,
    transformación y carga.
    """

    print("Iniciando pipeline ETL...")

    df = extraer_datos(ruta_entrada)
    df = transformar_datos(df)
    cargar_datos(df, ruta_salida)

    print("Pipeline ETL finalizado correctamente.")
'''

pipeline_path = PROJECT_PATH / "src" / "pipeline.py"

with open(pipeline_path, "w", encoding="utf-8") as archivo:
    archivo.write(pipeline_code)

print("pipeline.py corregido correctamente.")

pipeline.py corregido correctamente.


In [44]:
%cd $PROJECT_PATH
!python main.py

/content/drive/MyDrive/Data Science/Data Science II/preentrega-5-pipeline-etl
Iniciando pipeline ETL...
Extracción completada: 700 filas y 12 columnas.
Duplicados eliminados: 8
Fechas inválidas eliminadas: 0
Valores nulos restantes: 0
Archivo Parquet generado en: output/datos_procesados.parquet
Pipeline ETL finalizado correctamente.


In [45]:
archivo_parquet = PROJECT_PATH / "output" / "datos_procesados.parquet"

df_procesado = pd.read_parquet(archivo_parquet)

print("VALIDACIÓN DEL PIPELINE")
print("-----------------------")
print(f"Filas originales: {df.shape[0]}")
print(f"Filas procesadas: {df_procesado.shape[0]}")
print(f"Columnas: {df_procesado.shape[1]}")
print(f"Duplicados: {df_procesado.duplicated().sum()}")
print(f"Valores nulos: {df_procesado.isna().sum().sum()}")

display(df_procesado.head())

VALIDACIÓN DEL PIPELINE
-----------------------
Filas originales: 700
Filas procesadas: 692
Columnas: 12
Duplicados: 0
Valores nulos: 0


,id,reason_for_absence,date,transportation_expense,distance_to_work,age,daily_work_load_average,body_mass_index,education,children,pets,absenteeism_time_in_hours
0,11,26,2015-07-07,289,36,33,239.554001,30,1,2,1,4
1,36,0,2015-07-14,118,13,50,239.554001,31,1,1,0,0
2,3,23,2015-07-15,179,51,38,239.554001,31,1,0,0,2
3,7,7,2015-07-16,279,5,39,239.554001,24,1,2,0,4
4,11,23,2015-07-23,289,36,33,239.554001,30,1,2,1,2


### 10.1 Resultados de la validación

La ejecución del pipeline ETL finalizó correctamente.

El dataset original contenía **700 registros y 12 variables**. Durante la etapa de transformación se eliminaron **8 registros duplicados**, obteniendo como resultado un dataset procesado de **692 registros y 12 variables**.

La validación final confirmó que:

- no existen registros duplicados;
- no quedaron valores nulos sin tratar;
- no se detectaron fechas inválidas;
- los nombres de las columnas fueron normalizados;
- el archivo procesado fue generado correctamente en formato Parquet.

Por lo tanto, el pipeline realiza correctamente las etapas de extracción, transformación y carga definidas para el proyecto.

## 11. Carga y elección del formato Parquet

La última etapa del pipeline corresponde a la carga de los datos procesados.

El resultado se almacena en formato **Parquet**, un formato de almacenamiento columnar diseñado para trabajar de manera eficiente con datos estructurados.

Se eligió Parquet porque:

- permite una mayor compresión que formatos de texto como CSV;
- conserva los tipos de datos;
- permite acceder eficientemente a columnas específicas;
- reduce el espacio necesario para almacenar grandes volúmenes de información;
- resulta adecuado para procesos analíticos y de Machine Learning.

Estas características permiten que el dataset procesado pueda ser reutilizado eficientemente en etapas posteriores de análisis y modelado.

## 12. Buenas prácticas del repositorio

Para evitar almacenar archivos innecesarios o pesados en GitHub se utilizará un archivo `.gitignore`.

Se excluirán del repositorio el dataset original, los archivos generados por el pipeline, archivos temporales de Python y entornos virtuales.

De esta forma el repositorio contendrá únicamente el código y la documentación necesarios para reproducir el proyecto.

In [46]:
gitignore = """# Dataset original
data/

# Archivos generados por el pipeline
output/

# Archivos temporales de Python
__pycache__/
*.pyc

# Entornos virtuales
.venv/
venv/

# Jupyter
.ipynb_checkpoints/
"""

with open(PROJECT_PATH / ".gitignore", "w") as archivo:
    archivo.write(gitignore)

print(".gitignore creado correctamente.")

.gitignore creado correctamente.


## 13. Documentación y reproducibilidad

Finalmente se creará el archivo `README.md`.

Este documento permitirá que cualquier persona que acceda al repositorio pueda comprender el objetivo del proyecto, instalar las dependencias necesarias y ejecutar el pipeline.

El README documentará además las etapas de extracción, transformación y carga desarrolladas.

In [47]:
readme = """# Pipeline ETL - Ausentismo Laboral

## Objetivo

Este proyecto implementa un pipeline ETL (Extract, Transform, Load) utilizando Python y Pandas para automatizar el procesamiento de un dataset de ausentismo laboral.

El pipeline permite extraer los datos desde un archivo CSV, aplicar reglas de limpieza y transformación y almacenar el resultado en formato Parquet.

## Estructura del proyecto

preentrega-5-pipeline-etl/
- main.py
- requirements.txt
- README.md
- .gitignore
- src/
  - pipeline.py
- data/
- output/

## Etapas del pipeline

### 1. Extracción

El archivo CSV es leído utilizando Pandas. Durante el procesamiento se optimizan los tipos de datos numéricos para reducir el uso de memoria.

### 2. Transformación

La etapa de transformación realiza las siguientes tareas:

- Normalización de nombres de columnas.
- Conversión de la variable de fecha.
- Eliminación de registros duplicados.
- Tratamiento de posibles valores nulos.
- Optimización de tipos numéricos.

Para posibles valores faltantes en variables numéricas se utiliza la mediana, debido a que es menos sensible a valores extremos que la media.

Aunque el dataset utilizado actualmente no presenta valores nulos, esta lógica se incorpora para que el pipeline pueda manejar futuros archivos que sí los contengan.

### 3. Carga

Los datos procesados se exportan en formato Parquet.

Parquet utiliza almacenamiento columnar, permite una buena compresión, conserva los tipos de datos y facilita el acceso eficiente a las columnas. Estas características lo hacen adecuado para procesos de análisis de datos y Machine Learning.

## Instalación

Clonar el repositorio e instalar las dependencias con el siguiente comando:

    pip install -r requirements.txt

## Ejecución

El archivo CSV de entrada debe ubicarse en:

    data/dataset.csv

Para ejecutar el pipeline:

    python main.py

El resultado será generado automáticamente en:

    output/datos_procesados.parquet

## Dependencias

- pandas
- pyarrow

## Resultado obtenido

El dataset original utilizado contiene 700 registros y 12 variables.

Durante la transformación se eliminaron 8 registros duplicados, obteniendo un dataset procesado final de 692 registros y 12 variables.

La validación final confirmó que el resultado no contiene registros duplicados ni valores nulos.

## Buenas prácticas

Se utiliza un archivo .gitignore para evitar incluir en el repositorio el dataset original, los archivos generados por el pipeline, archivos temporales de Python y entornos virtuales.
"""

readme_path = PROJECT_PATH / "README.md"

with open(readme_path, "w", encoding="utf-8") as archivo:
    archivo.write(readme)

print("README.md creado correctamente.")
print(f"Ubicación: {readme_path}")

README.md creado correctamente.
Ubicación: /content/drive/MyDrive/Data Science/Data Science II/preentrega-5-pipeline-etl/README.md


In [48]:
archivos_requeridos = [
    "main.py",
    "src/pipeline.py",
    "requirements.txt",
    "README.md",
    ".gitignore",
    "output/datos_procesados.parquet"
]

print("VERIFICACIÓN FINAL")
print("------------------")

for archivo in archivos_requeridos:
    ruta = PROJECT_PATH / archivo

    estado = "OK" if ruta.exists() else "FALTA"

    print(f"{estado}: {archivo}")

VERIFICACIÓN FINAL
------------------
OK: main.py
OK: src/pipeline.py
OK: requirements.txt
OK: README.md
OK: .gitignore
OK: output/datos_procesados.parquet
